## ex02_joins.ipynb

In [14]:
import pandas as pd
import sqlite3

## Create a connection to the database using the sqlite3 library.

In [15]:
db_path = "../data/checking-logs.sqlite"
connection = sqlite3.connect(db_path)

## Create a new table called datamart in the database by joining the tables pageviews and checker using only one query.

In [16]:
query = """
CREATE TABLE IF NOT EXISTS datamart AS
SELECT
    c.uid,
    c.labname,
    MIN(c.timestamp) AS first_commit_ts,
    MIN(p.datetime) AS first_view_ts
FROM checker c
LEFT JOIN pageviews p ON c.uid = p.uid
WHERE c.status = 'ready'
    AND c.numTrials = 1
    AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND c.uid LIKE 'user_%'
GROUP BY c.uid, c.labname;
"""
connection.execute("DROP TABLE IF EXISTS datamart")
connection.execute(query)
connection.commit()

## Using Pandas methods, create two dataframes: test and control.

In [17]:
datamart = pd.read_sql("SELECT * FROM datamart", connection)

datamart['uid'] = datamart['uid'].astype(object)
datamart['labname'] = datamart['labname'].astype(object)

datamart['first_commit_ts'] = pd.to_datetime(datamart['first_commit_ts'])
datamart['first_view_ts'] = pd.to_datetime(datamart['first_view_ts'])

test = datamart[datamart['first_view_ts'].notna()].copy()
control = datamart[datamart['first_view_ts'].isna()].copy()

avg_first_view_ts = test['first_view_ts'].mean()
control['first_view_ts'] = avg_first_view_ts

test.to_sql('test', connection, if_exists='replace', index=True)
control.to_sql('control', connection, if_exists='replace', index=True)

81

## Close the connection.

In [18]:
connection.close()

## review

In [19]:
datamart.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    object        
 1   labname          140 non-null    object        
 2   first_commit_ts  140 non-null    datetime64[us]
 3   first_view_ts    59 non-null     datetime64[us]
dtypes: datetime64[us](2), object(2)
memory usage: 4.5+ KB


In [20]:
test.info()

<class 'pandas.DataFrame'>
Index: 59 entries, 0 to 114
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     object        
 1   labname          59 non-null     object        
 2   first_commit_ts  59 non-null     datetime64[us]
 3   first_view_ts    59 non-null     datetime64[us]
dtypes: datetime64[us](2), object(2)
memory usage: 2.3+ KB


In [21]:
control.info()

<class 'pandas.DataFrame'>
Index: 81 entries, 12 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[us]
 3   first_view_ts    81 non-null     datetime64[us]
dtypes: datetime64[us](2), object(2)
memory usage: 3.2+ KB


In [22]:
conn = sqlite3.connect(db_path)

In [23]:
test = pd.io.sql.read_sql('SELECT * FROM test', conn)

In [24]:
control = pd.io.sql.read_sql('SELECT * FROM control', conn)

In [25]:
conn.close()